# Introduction to Apache Spark: Philosophy, Ecosystem & Primitives

**Module**: 01_spark_fundamentals  
**Notebook**: 01_introduction_to_spark.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The foundational concepts of Apache Spark, why it replaced Hadoop MapReduce, and its unified computational engine.
- **Why does this topic exist?**: Understanding Spark's design philosophy clarifies why DataFrames are faster than Python scripts and how distributed execution works.
- **What problem does it solve?**: Traditional tools (Pandas, raw Python, MapReduce) fail to scale beyond single-machine RAM or incur massive disk serialization bottlenecks.
- **Where is it used in real-world Data Engineering?**: Used at Netflix, Uber, Apple, and banks for petabyte-scale ETL, fraud detection, and recommendation pipelines.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 1–2 h |
| **Prerequisites** | None |
| **Required depth** | Conceptual understanding: can explain why Spark exists and when not to use it |
| **Data Engineer** | Choose Spark for the right workloads and explain the batch/streaming/SQL ecosystem. |
| **Senior Data Engineer** | Compare Spark with MapReduce, pandas, DuckDB, Flink, and a warehouse. |
| **Spark specialist** | Explain the design choices behind the unified engine and its execution model. |
| **Next topic** | `01_spark_fundamentals/02_spark_architecture.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Before Spark, iterative batch jobs often materialized intermediate results to disk between MapReduce stages.
That made machine-learning loops, graph algorithms, and interactive analysis slow. Spark introduced a unified
engine with lazy DAG execution and in-memory-friendly abstractions, but it is not automatically faster for small
data or transactional workloads.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **RDD** | A distributed collection you can transform | Immutable, partitioned, lineage-based fault tolerance |
| **DataFrame** | A distributed table with a schema | Catalyst-optimized structured API |
| **Spark SQL** | SQL over distributed data | Same engine as DataFrames, with a catalog and optimizer |
| **Lazy evaluation** | Build a plan, then run it | Transformations are recorded; actions trigger jobs |
| **DAG** | A graph of dependent stages | Scheduler splits at shuffle boundaries |
| **Cluster manager** | The resource broker | Standalone, YARN, Kubernetes, or a managed service |

### How it works, step by step

1. A user program creates a `SparkSession` on a driver.
2. Transformations build a logical plan; no data is processed yet.
3. An action submits a job to the DAGScheduler.
4. The DAG is split into stages at wide dependencies.
5. Tasks run on executors, one task per partition.
6. Results return to the driver or are written to storage.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. The Evolution of Big Data Processing: MapReduce vs Spark

```
Hadoop MapReduce (Disk-Bound Iteration):
[Input] -> [Map] -> [Disk Spill] -> [Network Shuffle] -> [Reduce] -> [Disk Write]
            |                                                           |
            +----------------- Repeat for Next Job <--------------------+
            (Massive Disk I/O Bottleneck at Every Iteration)

Apache Spark (In-Memory DAG Pipeline):
[Input] -> [Transform 1] -> [Transform 2] -> [In-Memory Cache] -> [Action Output]
            \__________________________/
              Fused into single Stage
               (Often faster for iterative workloads; the multiplier varies)
```

Spark's core innovations:
1. **In-Memory Computing**: Keeps intermediate data in memory when it fits, and spills to disk when it does not.
2. **DAG (Directed Acyclic Graph)**: Plans multi-step pipelines holistically before running.
3. **Unified Engine**: One unified API for SQL, streaming, machine learning, and graphs.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spark_Introduction_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Hands-On: Classic Distributed Word Count

Let us run the classic distributed word count on `/opt/spark-data/words.txt`.

In [ ]:
# Read text file as RDD
lines_rdd = spark.sparkContext.textFile("/opt/spark-data/words.txt")
print(f"Total lines in words.txt: {lines_rdd.count()}")

# Tokenize, map, and reduce
word_counts = (
    lines_rdd
    .flatMap(lambda line: line.lower().split())
    .map(lambda word: (word.strip(".,!?\"'"), 1))
    .filter(lambda pair: len(pair[0]) > 0)
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda pair: pair[1], ascending=False)
)

print("Top 10 most frequent words:")
for word, count in word_counts.take(10):
    print(f"  {word:15}: {count}")

## 4. The Same Problem in Spark SQL / DataFrame API

Now let us execute the exact same analysis using the DataFrame API and compare readability and optimization.

In [ ]:
from pyspark.sql import functions as F

words_df = spark.read.text("/opt/spark-data/words.txt")

df_counts = (
    words_df
    .select(F.explode(F.split(F.lower(F.col("value")), "\\s+")).alias("raw_word"))
    .select(F.regexp_replace(F.col("raw_word"), "[^a-zA-Z]", "").alias("word"))
    .filter(F.length(F.col("word")) > 0)
    .groupBy("word")
    .count()
    .orderBy(F.desc("count"))
)

print("DataFrame Word Count Output:")
df_counts.show(10)

---

## Experiment: RDD vs DataFrame Execution Plan Comparison

- **1. What are we testing?**: Compare how Spark optimizes DataFrame operations vs low-level RDD transformations.
- **2. What do we expect?**: DataFrame plan shows Catalyst optimization (WholeStageCodegen, HashAggregate), while RDD plan is opaque.

In [ ]:
print("=== DataFrame Physical Plan ===")
df_counts.explain()

print("\n=== RDD Lineage Debug String ===")
print(word_counts.toDebugString().decode('utf-8'))

**3. What happened?**: DataFrame execution plan shows optimized operators (*HashAggregate, *Project) whereas RDD shows raw stage steps.  
**4. Why did it happen?**: Catalyst optimizes DataFrame operations because it understands relations and column semantics.  
**5. Practical Engineering Lesson**: DataFrames allow Catalyst and Tungsten to optimize execution, whereas RDDs treat user functions as opaque black boxes.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Replacing a nightly MapReduce ETL

- **Industry**: Retail / e-commerce
- **Scenario**: A retailer joins orders, products, and customers every night to build a revenue mart.
- **Problem**: The MapReduce pipeline writes intermediate results between every stage, so a 3-hour SLA is hard to meet.
- **Why this concept is relevant**: Spark can pipeline narrow transformations, keep intermediate data in memory when useful, and express the join/aggregation in SQL or DataFrames.
- **Implementation idea**: Read raw files, clean and join in a DataFrame pipeline, write partitioned Parquet, and schedule the job with Airflow.
- **Expected result**: A shorter, more maintainable batch pipeline that produces the same revenue mart.
- **Alternative**: A cloud data warehouse with SQL transformations (for example, dbt plus a warehouse).
- **Trade-offs**: Spark is better when the data is large, semi-structured, or already in a data lake; a warehouse is simpler when the data fits and SQL is the main interface.

### Use Case 2: Unified batch and streaming analytics

- **Industry**: IoT / logistics
- **Scenario**: A logistics company processes historical delivery data in batch and live vehicle events in streaming.
- **Problem**: Two separate stacks increase operational cost and create inconsistent business logic.
- **Why this concept is relevant**: Spark Structured Streaming uses the same DataFrame/SQL API as batch, so the team can share transformations and run the same engine.
- **Implementation idea**: Land events in object storage, use Structured Streaming for live delay alerts, and run batch jobs for historical KPIs.
- **Expected result**: One platform and one programming model for both latency profiles.
- **Alternative**: Kafka Streams or Flink for true event-at-a-time processing.
- **Trade-offs**: Spark is convenient when batch and streaming share code; Flink is stronger for very low-latency stateful streaming.

---

## Business Value

- Can reduce time-to-insight for large, multi-source analytical pipelines.
- Can lower operational complexity by unifying batch, SQL, and streaming APIs.
- Supports better decisions when data is too large for a single machine.
- Does not automatically reduce cost; cluster size, shuffles, and storage still dominate.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Start with the smallest engine that solves the problem; do not use Spark for a few thousand rows.
- Measure job duration, shuffle bytes, spill, and cost before and after migration.
- Separate batch and streaming SLAs; do not assume the same latency target.
- Plan for failure: retries, idempotent writes, and backfills.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Measure the cost of starting a Spark job

**Steps**

1. Time a trivial `spark.range(1000).count()` action.
2. Compare it with a local Python loop over the same 1,000 integers.
3. Open the Spark UI Jobs tab and count the jobs and tasks.

**What to observe**: A tiny Spark job still pays driver, scheduler, and task-launch overhead.

**Healthy result**: You can explain why Spark is not automatically faster for small data.

**Problematic result**: Assuming Spark is always faster leads to unnecessary infrastructure and slower small jobs.

**Engineering takeaway**: Choose Spark based on data size, distribution, and SLA, not on hype.

---

## Trade-offs and Alternatives

### Advantages

- Unified API for batch, SQL, streaming, and ML
- Scales beyond one machine
- Large ecosystem and community

### Limitations

- JVM and scheduling overhead
- Not a transactional database
- Operational complexity for large clusters

### When to use it

- Large ETL
- Data lakehouse transformations
- Distributed analytics and feature preparation

### When not to use it

- Small pandas-sized datasets
- Sub-second API serving
- OLTP workloads

### Alternatives inside Spark

- DataFrames vs RDDs
- Spark SQL
- Structured Streaming

### Alternatives outside Spark

- pandas/DuckDB
- A cloud warehouse
- Flink
- Kafka Streams
- dbt

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Filter High-Frequency Words
**Problem Statement**: Filter df_counts to keep only words appearing 2 or more times and print the count of distinct words.  
**Input / Context**: df_counts DataFrame from earlier.

In [ ]:
# Exercise 1: Filter words with count >= 2

### Exercise 2 (Intermediate): Calculate Average Word Length
**Problem Statement**: Using DataFrame API, calculate the average length of all distinct words in words.txt.  
**Input / Context**: df_counts DataFrame

In [ ]:
# Exercise 2: Average word length

### Exercise 3 (Advanced): Monte Carlo Pi Estimation with Parallelize
**Problem Statement**: Estimate Pi using 1,000,000 random points distributed across RDD partitions.  
**Input / Context**: Random sampling with math.hypot(x, y) <= 1.0

In [ ]:
# Exercise 3: Distributed Pi calculation

---

## Production Debugging Exercise

### The over-engineered small job

**Scenario**: A team moves a 20 MB CSV aggregation from pandas to a 10-node Spark cluster and the job becomes slower and more expensive.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Measure the single-machine runtime first.
2. Check how much time is spent in job startup versus computation.
3. Consider DuckDB or a warehouse for data that fits on one node.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Filter High-Frequency Words):
```python
frequent_words = df_counts.filter(F.col("count") >= 2)
print("Count of frequent words:", frequent_words.count())
frequent_words.show()
```

#### Solution 2 (Calculate Average Word Length):
```python
avg_len = df_counts.select(F.avg(F.length(F.col("word"))).alias("avg_length")).collect()[0][0]
print(f"Average distinct word length: {avg_len:.2f}")
```

#### Solution 3 (Monte Carlo Pi Estimation with Parallelize):
```python
import random

def sample_point(_):
    x, y = random.random(), random.random()
    return 1 if (x*x + y*y) <= 1.0 else 0

SAMPLES = 500000
count_inside = spark.sparkContext.parallelize(range(SAMPLES), 8).map(sample_point).reduce(lambda a, b: a + b)
pi_estimate = 4.0 * count_inside / SAMPLES
print(f"Estimated Pi: {pi_estimate:.5f} (True Pi: 3.14159)")
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Spark Paradigm**: Spark eliminates disk bottlenecks by pipelining transformations in memory using DAG scheduling.
- **Unified API**: DataFrames provide a single high-level abstraction optimized by Catalyst across batch, streaming, and SQL.
- **RDD vs DataFrame**: Prefer DataFrames in production; use RDDs only for custom unstructured data manipulation.

### Essential Technical Terms
- `RDD`: Resilient Distributed Dataset (immutable, distributed collection of objects).
- `DAG`: Directed Acyclic Graph representing the dependency graph of transformations.
- `Action`: Terminal operation triggering execution across the cluster.

### Common Engineering Mistakes
- **Using RDDs for Tabular Data**: Misses Catalyst predicate pushdown, column pruning, and Tungsten code generation.
- **Imperative Python Loops**: Iterating over DataFrame rows with Python `for` loops bypasses distributed execution.

### Interview Questions on This Topic
- **Q: Why is Spark faster than MapReduce?**
  - *A*: Spark retains intermediate data in memory across stages and fuses narrow transformations into single tasks using DAG pipelining.
- **Q: What is an RDD?**
  - *A*: An immutable, lazily evaluated, partitioned collection of records that can be recomputed upon node failure via its lineage.




### More Interview Questions

- **Q: Why was Spark created?**
  - *A*: To address the disk materialization and limited API of MapReduce, especially for iterative and interactive workloads. It is not a universal replacement for every data system.
- **Q: When should you not use Spark?**
  - *A*: When the data fits comfortably on one machine, when sub-second latency is required, or when the workload needs OLTP transactions.

### What to Study Next
Open 01_spark_fundamentals/02_spark_architecture.ipynb to dissect the Driver, Master, Workers, and Executors.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')